In [2]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
from sklearn.model_selection import train_test_split

**DATASET - RAVDESS**

In [3]:
Ravdess = "dataset\\ravdess\\"

data = []

emotion_map = {
    1: "Neutral",
    2: "Calm",
    3: "Happy",
    4: "Sad",
    5: "Angry",
    6: "Fearful",
    7: "Disgust",
    8: "Surprised",
}

for dir in os.listdir(Ravdess):
    for file in os.listdir(os.path.join(Ravdess, dir)):
        filename, _ = os.path.splitext(file)
        parts = filename.split("-")
        emotion_code = int(parts[2])
        emotion = emotion_map.get(emotion_code, "Unknown")
        path = os.path.join(Ravdess, dir, file)
        data.append([path, emotion])

Ravdess_df = pd.DataFrame(data, columns=["path", "emotion"])

print(Ravdess_df.shape[0])
unique_labels = Ravdess_df["emotion"].unique()
print("Unique emotion labels:", unique_labels)
print(Ravdess_df["emotion"].value_counts())
Ravdess_df.head()

1440
Unique emotion labels: <StringArray>
['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust',
 'Surprised']
Length: 8, dtype: str
emotion
Calm         192
Happy        192
Sad          192
Angry        192
Fearful      192
Disgust      192
Surprised    192
Neutral       96
Name: count, dtype: int64


,path,emotion
0,dataset\ravdess\Actor_01\03-01-01-01-01-01-01.wav,Neutral
1,dataset\ravdess\Actor_01\03-01-01-01-01-02-01.wav,Neutral
2,dataset\ravdess\Actor_01\03-01-01-01-02-01-01.wav,Neutral
3,dataset\ravdess\Actor_01\03-01-01-01-02-02-01.wav,Neutral
4,dataset\ravdess\Actor_01\03-01-02-01-01-01-01.wav,Calm


**Output Dataframe to CSV**

In [8]:
def create_csv(df: pd.DataFrame, filename: str):
    csv_dir = "CSVs\\"

    if not os.path.exists(csv_dir):
        os.makedirs(csv_dir)
        print(f"Directory {csv_dir} created.")

    df.to_csv(os.path.join(csv_dir, filename), index=False)

    print(f"CSV file created at: {os.path.join(csv_dir, filename)}")


create_csv(Ravdess_df, "ravdess.csv")

CSV file created at: CSVs\ravdess.csv


**Silence Removal**

In [4]:
def remove_silence(df: pd.DataFrame, top_db=20):
    for audio in df["path"]:
        y, sr = librosa.load(audio, sr=None)

        intervals = librosa.effects.split(y, top_db=top_db)

        audio_trimmed = np.concatenate([y[start:end] for start, end in intervals])

        return audio_trimmed, sr

**Dataset partitioning**

In [ ]:
X = Ravdess_df["path"]
y = Ravdess_df["emotion"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

create_csv(pd.DataFrame({"path": X_train, "emotion": y_train}), "ravdess_train.csv")
create_csv(pd.DataFrame({"path": X_test, "emotion": y_test}), "ravdess_test.csv")

CSV file created at: CSVs\ravdess_train.csv
CSV file created at: CSVs\ravdess_test.csv
Training set size: 1152
Testing set size: 288
